# Lesson 1.3 — Configuration Management

**Level: Beginner**

## Learning Objectives
- Load environment variables from `.env` files with `python-dotenv`
- Parse and write YAML configuration with `PyYAML`
- Work with JSON configs using Python's built-in `json` module
- Apply the base-config + environment-overlay pattern
- Build a typed `AppConfig` dataclass

**Produces:** `config_loader.py` — a multi-environment config loader.

*Official refs:*
- *[python-dotenv](https://saurabh-kumar.com/python-dotenv/)*
- *[PyYAML](https://pyyaml.org/wiki/PyYAMLDocumentation)*

## Concepts

### The 12-Factor App Rule for Config

> *"Store config in the environment"* — [12factor.net/config](https://12factor.net/config)

This means:
- **Secrets** → `.env` files (or a secret manager — covered in Phase 4)
- **Structured non-secret config** → YAML/JSON files
- **Never** hardcode environment-specific values (database URLs, API endpoints) in code

### Config Priority Stack (higher overrides lower)

```
Hardcoded defaults
    ↓
Base config YAML       (configs/base.yaml)
    ↓
Env-specific YAML      (configs/staging.yaml)
    ↓
Environment variables  (.env file or shell)    ← highest priority
```

This pattern is used by Kubernetes (Kustomize base + overlay), Helm (values.yaml + env values), and most frameworks.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "python-dotenv", "pyyaml", "--quiet"], capture_output=True)

import os, json, yaml
from pathlib import Path
from dataclasses import dataclass, field
from dotenv import load_dotenv, dotenv_values

print(f"pyyaml version   : {yaml.__version__}")
print(f"dotenv installed : OK")

## Code-Along: Step 1 — python-dotenv

In [ ]:
# Write a sample .env file
env_file = Path("sample.env")
env_file.write_text(
    "# App config\n"
    "ENVIRONMENT=development\n"
    "LOG_LEVEL=DEBUG\n"
    "AWS_DEFAULT_REGION=us-east-1\n"
    "SLACK_CHANNEL=#ops-alerts\n"
)

# load_dotenv() reads the file and injects values into os.environ
load_dotenv(env_file, override=True)

print("Loaded from .env:")
for key in ["ENVIRONMENT", "LOG_LEVEL", "AWS_DEFAULT_REGION", "SLACK_CHANNEL"]:
    print(f"  {key:<25} = {os.environ.get(key)}")

# dotenv_values() reads WITHOUT touching os.environ (safe for testing)
vals = dotenv_values(env_file)
print(f"\ndotenv_values keys: {list(vals.keys())}")
env_file.unlink()

## Code-Along: Step 2 — YAML Configuration

In [ ]:
# Base config — shared across all environments
base = {
    "app": {"name": "ops-automation-platform", "version": "1.0.0"},
    "aws": {"region": "us-east-1", "services": ["ec2", "s3", "cloudwatch"]},
    "monitoring": {
        "prometheus_port": 9090,
        "scrape_interval_seconds": 15,
        "thresholds": {"cpu_pct": 80, "disk_pct": 85, "memory_pct": 90},
    },
    "logging": {"level": "INFO", "format": "json"},
}

# Write to disk
cfg_path = Path("config.base.yaml")
with open(cfg_path, "w") as f:
    yaml.dump(base, f, default_flow_style=False, sort_keys=False)

print("Written config.base.yaml:")
print(cfg_path.read_text())
cfg_path.unlink()

In [ ]:
# Environment overlays — only the fields that differ
overlays = {
    "development": {
        "monitoring": {"scrape_interval_seconds": 5},   # faster in dev
        "logging": {"level": "DEBUG"},
    },
    "production": {
        "monitoring": {"scrape_interval_seconds": 30},
        "logging": {"level": "WARNING"},
    },
}

def deep_merge(base: dict, override: dict) -> dict:
    """Recursively merge override into base (override wins)."""
    result = base.copy()
    for k, v in override.items():
        if k in result and isinstance(result[k], dict) and isinstance(v, dict):
            result[k] = deep_merge(result[k], v)
        else:
            result[k] = v
    return result

env = os.environ.get("ENVIRONMENT", "development")
final = deep_merge(base, overlays.get(env, {}))

print(f"Final config for '{env}':")
print(f"  scrape_interval_seconds : {final['monitoring']['scrape_interval_seconds']}")
print(f"  log level               : {final['logging']['level']}")

## Code-Along: Step 3 — Typed AppConfig with Dataclasses

In [ ]:
@dataclass
class AWSConfig:
    region: str = "us-east-1"
    account_id: str = ""

@dataclass
class MonitoringConfig:
    prometheus_port: int = 9090
    scrape_interval_seconds: int = 15
    cpu_threshold: float = 80.0
    disk_threshold: float = 85.0

@dataclass
class AppConfig:
    environment: str = "development"
    log_level: str = "INFO"
    aws: AWSConfig = field(default_factory=AWSConfig)
    monitoring: MonitoringConfig = field(default_factory=MonitoringConfig)
    slack_webhook_url: str = ""

def load_config() -> AppConfig:
    """Load AppConfig from environment variables."""
    return AppConfig(
        environment=os.environ.get("ENVIRONMENT", "development"),
        log_level=os.environ.get("LOG_LEVEL", "INFO"),
        aws=AWSConfig(
            region=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
            account_id=os.environ.get("AWS_ACCOUNT_ID", ""),
        ),
        monitoring=MonitoringConfig(
            prometheus_port=int(os.environ.get("PROMETHEUS_PORT", "9090")),
            scrape_interval_seconds=int(os.environ.get("SCRAPE_INTERVAL", "15")),
        ),
        slack_webhook_url=os.environ.get("SLACK_WEBHOOK_URL", ""),
    )

cfg = load_config()
print("AppConfig:")
print(f"  environment        : {cfg.environment}")
print(f"  log_level          : {cfg.log_level}")
print(f"  aws.region         : {cfg.aws.region}")
print(f"  prometheus_port    : {cfg.monitoring.prometheus_port}")
print(f"  scrape_interval    : {cfg.monitoring.scrape_interval_seconds}s")
print(f"  slack configured   : {'Yes' if cfg.slack_webhook_url else 'No'}")

## Exercise

Build an environment-aware loader that merges YAML + JSON.

**Your task:**
1. Write a base YAML to `config.base.yaml`  (keys: `app_name`, `replicas`, `log_level`)
2. Write a staging overlay to `config.staging.json` (overrides: `replicas=2`, `log_level=WARNING`)
3. Implement `load_config(env: str) -> dict` that deep-merges them
4. Test with `load_config("staging")`

In [ ]:
# TODO: write config.base.yaml
# TODO: write config.staging.json
# TODO: implement load_config(env) -> dict
def load_config_env(env: str) -> dict:
    pass

result = load_config_env("staging")
print(result)
# Expected: {'app_name': '...', 'replicas': 2, 'log_level': 'WARNING'}


## Key Takeaways
- `python-dotenv` injects `.env` into `os.environ` — same interface as CI/CD secrets
- YAML is the DevOps-standard human-readable config format; JSON for machine-generated payloads
- Base-config + overlay pattern is used in Kubernetes (Kustomize), Helm, and Terraform
- Python dataclasses give you typed, IDE-friendly configuration objects

---
**Next:** [Lesson 1.4 — CLI Tools for Ops](lesson-1.4-cli-tools.ipynb)